
# Building and Learning a Bayesian Network

We use the binary Sprinkler network

`Cloudy -> Rain`, `Cloudy -> Sprinkler`, `Rain -> WetGrass`, `Sprinkler -> WetGrass`.

The joint distribution factorises as

\[
P(C,R,S,W)=P(C)P(R\mid C)P(S\mid C)P(W\mid R,S).
\]

The workflow is: **specify -> implement -> infer -> independently verify -> learn from data -> validate**.


In [ ]:

import itertools
import numpy as np
import pandas as pd

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator

np.random.seed(7)



## 1. Trusted Bayesian-network specification

Parameters:

- \(P(C=1)=0.5\)
- \(P(R=1\mid C=0)=0.2,\;P(R=1\mid C=1)=0.8\)
- \(P(S=1\mid C=0)=0.5,\;P(S=1\mid C=1)=0.1\)
- \(P(W=1\mid R,S)=0.01,0.90,0.90,0.99\) for parent states `(00,01,10,11)`.


In [ ]:

def build_reference_model():
    model = DiscreteBayesianNetwork([
        ("Cloudy", "Rain"),
        ("Cloudy", "Sprinkler"),
        ("Rain", "WetGrass"),
        ("Sprinkler", "WetGrass"),
    ])

    cpd_cloudy = TabularCPD("Cloudy", 2, [[0.5], [0.5]])

    cpd_rain = TabularCPD(
        "Rain", 2,
        [[0.8, 0.2],
         [0.2, 0.8]],
        evidence=["Cloudy"],
        evidence_card=[2],
    )

    cpd_sprinkler = TabularCPD(
        "Sprinkler", 2,
        [[0.5, 0.9],
         [0.5, 0.1]],
        evidence=["Cloudy"],
        evidence_card=[2],
    )

    cpd_wetgrass = TabularCPD(
        "WetGrass", 2,
        [[0.99, 0.10, 0.10, 0.01],
         [0.01, 0.90, 0.90, 0.99]],
        evidence=["Rain", "Sprinkler"],
        evidence_card=[2, 2],
    )

    model.add_cpds(cpd_cloudy, cpd_rain, cpd_sprinkler, cpd_wetgrass)
    assert model.check_model()
    return model


reference_model = build_reference_model()
print("Model valid:", reference_model.check_model())
print("Edges:", sorted(reference_model.edges()))

for cpd in reference_model.get_cpds():
    print()
    print(cpd)



## 2. Exact inference

Compute \(P(Rain=1\mid WetGrass=1)\) using variable elimination.


In [ ]:

inference = VariableElimination(reference_model)
posterior_rain = inference.query(
    variables=["Rain"],
    evidence={"WetGrass": 1},
    show_progress=False,
)
print(posterior_rain)
pgmpy_value = float(posterior_rain.values[1])
print("P(Rain=1 | WetGrass=1) =", pgmpy_value)


## 3. Independent verification by enumerating all 16 assignments

A successful program run is not by itself evidence that the probabilistic model is correct.

In [ ]:

def p_cloudy(c):
    return 0.5

def p_rain(r, c):
    p_true = {0: 0.2, 1: 0.8}[c]
    return p_true if r else 1 - p_true

def p_sprinkler(s, c):
    p_true = {0: 0.5, 1: 0.1}[c]
    return p_true if s else 1 - p_true

def p_wetgrass(w, r, s):
    p_true = {
        (0, 0): 0.01,
        (0, 1): 0.90,
        (1, 0): 0.90,
        (1, 1): 0.99,
    }[(r, s)]
    return p_true if w else 1 - p_true

numerator = denominator = 0.0
for c, r, s, w in itertools.product([0, 1], repeat=4):
    joint = p_cloudy(c) * p_rain(r, c) * p_sprinkler(s, c) * p_wetgrass(w, r, s)
    if w == 1:
        denominator += joint
        if r == 1:
            numerator += joint

enumerated_value = numerator / denominator
print("Enumeration:", enumerated_value)
print("pgmpy     :", pgmpy_value)
assert np.isclose(enumerated_value, pgmpy_value, atol=1e-12)
print("Independent posterior check: PASS")



## 4. Parameter estimation from data

Keep the graph fixed, generate observations from the trusted model, and estimate CPTs from data.


In [ ]:

data = reference_model.simulate(n_samples=1000, seed=7, show_progress=False)
print(data.head())
print("Shape:", data.shape)

mle_model = DiscreteBayesianNetwork(list(reference_model.edges()))
mle_model.fit(data, estimator=MaximumLikelihoodEstimator)
assert mle_model.check_model()

print("\nMLE estimate for Rain:")
print(mle_model.get_cpds("Rain"))



## 5. Sparse data: MLE versus Bayesian estimation

With small data, MLE can produce unstable extreme probabilities. A Bayesian estimator adds prior pseudo-counts and regularises the estimate.


In [ ]:

small_data = reference_model.simulate(n_samples=30, seed=11, show_progress=False)

mle_small = DiscreteBayesianNetwork(list(reference_model.edges()))
mle_small.fit(small_data, estimator=MaximumLikelihoodEstimator)

bayes_small = DiscreteBayesianNetwork(list(reference_model.edges()))
bayes_small.fit(
    small_data,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10,
)

assert mle_small.check_model()
assert bayes_small.check_model()

print("MLE Rain CPD")
print(mle_small.get_cpds("Rain"))
print("\nBayesian Rain CPD")
print(bayes_small.get_cpds("Rain"))



## 6. Optional: use a local coding LLM as an engineering assistant

The LLM's job is **only to translate the already-specified BN into code**. `pgmpy` remains the probabilistic modelling/inference library, and the generated code must be inspected and tested independently.

The prompt below is deliberately explicit. The model download is optional and may require several GB.


In [ ]:

CODER_PROMPT = """
Write concise Python code using the current pgmpy API.

Construct this binary Bayesian network:
Cloudy -> Rain
Cloudy -> Sprinkler
Rain -> WetGrass
Sprinkler -> WetGrass

Use these probabilities:
P(Cloudy=1)=0.5
P(Rain=1|Cloudy=0)=0.2
P(Rain=1|Cloudy=1)=0.8
P(Sprinkler=1|Cloudy=0)=0.5
P(Sprinkler=1|Cloudy=1)=0.1
P(WetGrass=1|Rain,Sprinkler) for (00,01,10,11) is
0.01, 0.90, 0.90, 0.99.

Use DiscreteBayesianNetwork and TabularCPD.
Call check_model().
Then use VariableElimination to compute P(Rain | WetGrass=1).
Store the model in generated_model and posterior in generated_posterior.
Do not read/write files or use network access.
Return only Python code.
"""
print(CODER_PROMPT)


In [ ]:

# Optional. Leave False if you only need the probabilistic lab itself.
USE_LOCAL_LLM = False

if USE_LOCAL_LLM:
    from transformers import pipeline

    llm = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-Coder-1.5B-Instruct",
        device_map="auto",
        dtype="auto",
    )
    messages = [
        {"role": "system", "content": "Return only safe, concise Python code using pgmpy."},
        {"role": "user", "content": CODER_PROMPT},
    ]
    response = llm(messages, max_new_tokens=1200, do_sample=False)
    print(response[0]["generated_text"][-1]["content"])
else:
    print("Local LLM step skipped; set USE_LOCAL_LLM=True only if you want to download/run it.")



## Validation summary

A valid implementation should satisfy all of the following:

- exact expected nodes and edges;
- `model.check_model()` succeeds;
- CPT columns represent normalized distributions;
- exact inference agrees with independent enumeration;
- learned models preserve the fixed graph structure;
- generated code is inspected before execution.

This separates **scientific correctness of the BN** from **software generation by an LLM**.
